# Cats vs Dogs, the borrowed-brain way (transfer learning)

In `catDog.ipynb` we built a picture-sorter from scratch and got it to about **85%**.
It learned everything from nothing: what an edge is, what fur looks like, what an ear is —
all from 8,000 photos. That is a lot to ask.

**Transfer learning** means borrowing a model that has already done that homework.
`ResNet18` was trained on ImageNet: 1.2 million photos across 1,000 everyday things
(cars, mushrooms, violins, and yes, many breeds of cat and dog). It already knows what
fur, whiskers, eyes and paws look like.

Think of it like hiring someone who has spent years looking at animals, and teaching them
one new thing: *"when you see this, say cat; when you see that, say dog."* You are not
retraining their eyes — you are only teaching the final decision.

So the plan is:

1. Load ResNet18 **with** its learned knowledge.
2. **Freeze** it, so training cannot damage what it already knows.
3. Chop off its last layer (which names 1,000 things) and bolt on our own (which names 2).
4. Train **only** that new last layer.

This trains far fewer numbers, so it is fast, and it usually lands around **97-98%** in
about 3 passes.

In [1]:
# ==========================================================
# 1. TOOLS AND SETTINGS
# ==========================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# Use the fastest chip available: NVIDIA GPU, else Apple's GPU, else the plain CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"using device- {device}")

# ResNet18 was trained on 224x224 pictures, so we feed it the same size.
IMG_SIZE = 224

# Only the new last layer learns, so a handful of passes is plenty.
epochs = 3

using device- mps


## 2. Preparing the photos

Same idea as before, with one rule added: **a borrowed model expects its food prepared
the way it was prepared during its own training.** ResNet18 was fed 224x224 pictures,
with colors shifted using those exact `mean` and `std` numbers below. Feed it something
prepared differently and it gets confused, the way a recipe fails if you swap salt for sugar.

Training photos still get small random changes (mirror, tilt) so the model sees variety.
Test photos stay clean, because grading has to be fair.

In [2]:
# ==========================================================
# 2. LOAD AND PREPARE THE PHOTOS
# ==========================================================

# These two lists are ImageNet's color settings - ResNet18 expects exactly these.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Step A: study photos - resized, plus small random changes for variety.
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),      # everything the same size
    transforms.RandomHorizontalFlip(),            # a mirrored cat is still a cat
    transforms.RandomRotation(10),                # tilt slightly, up to 10 degrees
    transforms.ToTensor(),                        # turn the picture into numbers
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Step B: exam photos - resized only, no random changes.
test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

# Step C: point at the folders. ImageFolder reads the labels off the folder names.
train_dataset = datasets.ImageFolder("archive/training_set/training_set", transform=train_transform)
test_dataset = datasets.ImageFolder("archive/test_set/test_set", transform=test_transform)

# Step D: hand the photos out in batches of 32.
# Shuffle the study photos so it never sees all cats then all dogs. No need to shuffle the exam.
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

print(f"{len(train_dataset)} training photos, {len(test_dataset)} test photos")
print(f"labels: {train_dataset.class_to_idx}")   # {'cats': 0, 'dogs': 1}

8005 training photos, 2023 test photos
labels: {'cats': 0, 'dogs': 1}


## 3. Borrow the brain, then give it a new mouth

Three small moves here, and each one matters:

**Load it with its memories.** `weights=ResNet18_Weights.DEFAULT` downloads (once) the
numbers ResNet18 learned from a million photos. Without that argument you get the same
shape of model but with random numbers inside — an empty brain, and none of the benefit.

**Freeze it.** `requires_grad = False` tells PyTorch *"do not adjust these numbers."*
Its knowledge stays locked. This is also why training is fast: there are 11 million numbers
in ResNet18 and we are going to adjust about a thousand of them.

**Swap the last layer.** ResNet18's last layer (`fc`, short for fully-connected) outputs
1,000 scores — one per ImageNet category. We throw it away and bolt on a fresh one with
2 outputs: cat and dog. A brand-new layer is unfrozen by default, so it is the only part
that learns.

In [3]:
# ==========================================================
# 3. BUILD THE MODEL OUT OF A BORROWED ONE
# ==========================================================

# Step A: load ResNet18 WITH everything it learned from 1.2 million photos.
# (First run downloads ~45MB, after that it is cached on your machine.)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Step B: freeze every single number in it - none of these will be changed by training.
for parameter in model.parameters():
    parameter.requires_grad = False

# Step C: find out how many numbers arrive at its last layer (512 for ResNet18).
how_many_inputs = model.fc.in_features

# Step D: replace the 1000-way last layer with our own 2-way one (cat, dog).
# This new layer is unfrozen by default, so it is the only part that will learn.
model.fc = nn.Linear(in_features=how_many_inputs, out_features=2)

model = model.to(device)

# Proof of how little we are actually training:
total = sum(p.numel() for p in model.parameters())
learning = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"numbers in the model: {total:,}")
print(f"numbers we will train: {learning:,}  ({learning / total * 100:.3f}%)")

numbers in the model: 11,177,538
numbers we will train: 1,026  (0.009%)


## 4. Teach it the one new thing

The training loop is exactly the one from `catDog.ipynb`, with a single difference:

```python
optim.Adam(model.fc.parameters(), lr=0.001)
```

`model.fc.parameters()` instead of `model.parameters()` — we hand the optimizer only the
new layer's numbers. It has nothing else it is allowed to touch.

As before, we grade after every pass and keep a copy of the best one, because the last
pass is not always the best pass.

In [4]:
# ==========================================================
# 4. TRAIN (only the new last layer) AND GRADE EACH PASS
# ==========================================================

loss_fun = nn.CrossEntropyLoss()                              # how wrong was each guess
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)       # ONLY the new layer learns


def check_score():
    """Grade the model on the test photos, give back the % it got right."""
    model.eval()                                     # exam mode
    correct = 0
    total = 0
    with torch.no_grad():                            # just guessing, no learning
        for images, labels in test_dataloader:
            images, labels = images.to(device), labels.to(device)
            guess = model(images)
            _, final = torch.max(guess, dim=1)       # take the higher of the 2 scores
            correct += (final == labels).sum().item()
            total += labels.size(0)
    model.train()                                    # back to learning mode
    return correct / total * 100


best_score = 0

model.train()
for epoch in range(epochs):
    total_loss = 0

    for images, labels in train_dataloader:
        images, labels = images.to(device), labels.to(device)   # send to the GPU/CPU

        optimizer.zero_grad()                  # wipe the last batch's corrections
        guess = model(images)                  # what does it think these are?
        loss = loss_fun(guess, labels)         # how wrong was it?
        loss.backward()                        # work out which way to nudge things
        optimizer.step()                       # nudge the new layer

        total_loss += loss.item()              # keep a running total to report

    score = check_score()
    print(f"Epoch {epoch + 1}: avg loss {total_loss / len(train_dataloader):.4f}  test {score:.1f}%")

    # keep a copy of the best model so far, so a bad later pass cannot spoil it
    if score > best_score:
        best_score = score
        torch.save(model.state_dict(), "cat_dog_resnet.pth")

print(f"\nbest test score- {best_score:.1f}%  (saved to cat_dog_resnet.pth)")
print(f"the from-scratch model in catDog.ipynb scored 84.8%")

Epoch 1: avg loss 0.1788  test 94.3%
Epoch 2: avg loss 0.1065  test 97.8%
Epoch 3: avg loss 0.0977  test 97.7%

best test score- 97.8%  (saved to cat_dog_resnet.pth)
the from-scratch model in catDog.ipynb scored 84.8%


## 5. Try it on one photo

The payoff. Hand it a single picture and get back an answer in English.

Two details worth knowing:

- `unsqueeze(0)` wraps the one photo in a batch of one. The model always expects a stack
  of photos, even when the stack is one tall.
- `softmax` turns the two raw scores into two percentages that add up to 100, so you get
  "92% sure it's a dog" instead of "the dog number was 4.1".

In [5]:
# ==========================================================
# 5. ASK IT ABOUT ONE PHOTO
# ==========================================================

from PIL import Image

# Folder names became the labels, in alphabetical order: cats = 0, dogs = 1.
names = train_dataset.classes


def what_is_it(photo_path):
    """Open one photo and say whether it is a cat or a dog, and how sure we are."""
    photo = Image.open(photo_path).convert("RGB")     # open it, force 3 color channels
    ready = test_transform(photo)                     # prepare it the same way as the exam photos
    batch_of_one = ready.unsqueeze(0).to(device)      # wrap it in a batch of one

    model.eval()
    with torch.no_grad():
        scores = model(batch_of_one)                  # two raw scores: cat, dog
        chances = torch.softmax(scores, dim=1)[0]     # turn them into percentages

    best = chances.argmax().item()                    # which one scored higher
    return names[best], chances[best].item() * 100


# Try it on a few photos the model has never seen.
for path in [
    "archive/test_set/test_set/cats/cat.4001.jpg",
    "archive/test_set/test_set/dogs/dog.4001.jpg",
]:
    answer, sureness = what_is_it(path)
    print(f"{path.split('/')[-1]:>16}  ->  {answer}  ({sureness:.1f}% sure)")

    cat.4001.jpg  ->  cats  (99.9% sure)
    dog.4001.jpg  ->  dogs  (68.0% sure)


## What just happened, and what to try next

You trained roughly **1,000 numbers** instead of 11 million, in 3 passes instead of 20,
and beat the from-scratch model by a wide margin. That is the whole point of transfer
learning: almost nobody builds an image model from scratch any more. They borrow one
and teach it the last step.

**If you want to push higher: fine-tuning.** Unfreeze the borrowed model's *last block*
and let it adjust a little, at a much smaller learning rate so its knowledge is nudged
rather than wrecked:

```python
for parameter in model.layer4.parameters():   # just the last block, not the whole thing
    parameter.requires_grad = True

optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 0.0001},   # gentle: it already knows a lot
    {"params": model.fc.parameters(), "lr": 0.001},        # normal: this one is still new
])
```

Then run a few more passes. This usually buys another percent or two. Use a small learning
rate — a big one on a pre-trained model wipes out exactly the knowledge you borrowed it for.

**Other models to borrow**, all loaded the same way, all with an `fc` or `classifier`
last layer to swap: `models.resnet50` (bigger, slower, more accurate),
`models.efficientnet_b0` (good accuracy for its size), `models.mobilenet_v3_small`
(built for phones).